In [1]:
import os
import json
from unstructured.partition.pdf import partition_pdf

/Users/aniruddhajoshi/hrag/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def parse_pdf_hierarchically(pdf_path, extra_metadata=None):  
    """
    Reads a PDF, detects its visual layout, and groups paragraphs 
    under their respective section headers.
    """
    if extra_metadata is None:
        extra_metadata = {}
        
    print(f" Processing: {os.path.basename(pdf_path)}... (This might take a minute on the first run)")
    
    try:
        elements = partition_pdf(
            filename=pdf_path, 
            strategy="hi_res",
            infer_bounding_boxes=True
        )
    except Exception as e:
        print(f"Error parsing {pdf_path}: {e}")
        return []

    document_chunks = []
    current_section = "Abstract / Introduction" # Default starting section
    
    for element in elements:
        element_type = element.category
        clean_text = str(element.text).replace("\n", " ").strip()
        
        # 1. Update our tracker if we hit a new Section Header or Title
        if element_type in ["Title", "Header"]:
            # Only update if the header is meaningful (not just a page number)
            if len(clean_text) > 3: 
                current_section = clean_text
                
        # 2. Extract Data Tables perfectly
        elif element_type == "Table":
            # Unstructured converts the table to HTML natively!
            table_html = element.metadata.text_as_html if hasattr(element.metadata, 'text_as_html') else clean_text
            metadata = {
                "source": os.path.basename(pdf_path),
                "section": current_section
            }
            metadata.update(extra_metadata)
            document_chunks.append({
                "type": "table",
                "text": table_html,
                "metadata": metadata
            })
            
        # 3. Extract Narrative Text and attach the current header
        elif element_type in ["NarrativeText", "Text", "ListItem"]:
            # Ignore tiny artifact strings (like isolated periods or single words)
            if len(clean_text) > 40:
                metadata = {
                    "source": os.path.basename(pdf_path),
                    "section": current_section
                }
                metadata.update(extra_metadata)
                document_chunks.append({
                    "type": "text",
                    "text": clean_text,
                    "metadata": metadata
                })
                
    return document_chunks

In [3]:
# --- Test the Pipeline ---
papers_dir = "./dataset"
metadata_file = "metadata.json"

# Make sure the directory exists
if not os.path.exists(papers_dir):
    print(f"Could not find the '{papers_dir}' directory.")
    exit()

# Load the external metadata mapping
with open(metadata_file, "r") as f:
    pdf_metadata_mapping = json.load(f)

# Get the PDFs in the folder
pdf_files = [f for f in os.listdir(papers_dir) if f.endswith('.pdf')]

if not pdf_files:
    print(" No PDFs found in the 'papers' folder!")
    exit()

all_chunks = []
for pdf_file in pdf_files:
    pdf_path = os.path.join(papers_dir, pdf_file)
    extra_meta = pdf_metadata_mapping.get(pdf_file, {})
    chunks = parse_pdf_hierarchically(pdf_path, extra_metadata=extra_meta)
    all_chunks.extend(chunks)

print(f"\n Successfully extracted {len(all_chunks)} contextual chunks across all documents!")
print("-" * 50)
print("Here is a sample of what your parsed data looks like:\n")

# Print a sample of the first 3 chunks to verify
for i, chunk in enumerate(all_chunks[5:8]): # Skipping first few in case of title pages
    print(json.dumps(chunk, indent=2))

# Save the parsed chunks to a JSON file for embedding later
with open("chunks.json", "w") as f:
    json.dump(all_chunks, f, indent=2)
print("\nSaved chunks to chunks.json")

 Processing: p1.pdf... (This might take a minute on the first run)
 Processing: p2.pdf... (This might take a minute on the first run)
 Processing: p3.pdf... (This might take a minute on the first run)

 Successfully extracted 526 contextual chunks across all documents!
--------------------------------------------------
Here is a sample of what your parsed data looks like:

{
  "type": "text",
  "text": "The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU